# COVID-19 Chest X-ray Classifier — Training (Azure ML)

**Goal:** classify chest X-rays into **Covid**, **Normal** and **Viral Pneumonia**.

**Method:** transfer learning with a **ResNet-50** pretrained on ImageNet. The early layers are frozen and the last block plus a new classifier head are fine-tuned on our images.

**Run this notebook on an Azure ML *compute instance*** with the kernel **"Python 3.10 - Pytorch and Tensorflow"** (or any kernel with PyTorch installed).

Steps: 1) setup → 2) load & explore data → 3) build model → 4) train → 5) evaluate on the test set → 6) save & register the model in Azure ML.

## 1. Setup

In [ ]:
%pip install -q azure-ai-ml azure-identity scikit-learn matplotlib seaborn

In [ ]:
import os, json, zipfile, random, time, copy
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# ---------- Configuration ----------
DATA_ZIP   = "Covid19-dataset.zip"   # upload the zip next to this notebook
DATA_DIR   = "Covid19-dataset"       # folder created after unzipping
OUTPUT_DIR = "outputs"
MODEL_DIR  = os.path.join(OUTPUT_DIR, "model")

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 15
LR         = 1e-4
VAL_SPLIT  = 0.15                    # 15% of the training images are held out for validation
SEED       = 42
PRETRAINED = True                    # use ImageNet weights (recommended)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)
os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Load and explore the data
The dataset is already split into `train/` and `test/`, each with one folder per class.

In [ ]:
if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(".")
    print("Unzipped", DATA_ZIP)

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

for split in [TRAIN_DIR, TEST_DIR]:
    counts = {c: len(os.listdir(os.path.join(split, c))) for c in sorted(os.listdir(split))}
    print(f"{split}: {counts}  (total {sum(counts.values())})")

In [ ]:
# Class distribution
train_counts = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in sorted(os.listdir(TRAIN_DIR))}
test_counts  = {c: len(os.listdir(os.path.join(TEST_DIR, c)))  for c in sorted(os.listdir(TEST_DIR))}

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(train_counts))
ax.bar(x - 0.2, train_counts.values(), 0.4, label="train")
ax.bar(x + 0.2, test_counts.values(),  0.4, label="test")
ax.set_xticks(x, train_counts.keys()); ax.set_ylabel("images"); ax.set_title("Images per class"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=120); plt.show()

In [ ]:
# A few sample images from each class
from PIL import Image
classes = sorted(os.listdir(TRAIN_DIR))
fig, axes = plt.subplots(len(classes), 4, figsize=(10, 7.5))
for r, c in enumerate(classes):
    files = sorted(os.listdir(os.path.join(TRAIN_DIR, c)))[:4]
    for k, f in enumerate(files):
        axes[r, k].imshow(Image.open(os.path.join(TRAIN_DIR, c, f)).convert("L"), cmap="gray")
        axes[r, k].axis("off")
    axes[r, 0].set_title(c, loc="left")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "samples.png"), dpi=100); plt.show()

### Preprocessing and augmentation
The training set is small (~250 images), so **augmentation** (small crops, rotations, brightness/contrast changes) helps the model generalise. Validation and test images are only resized and normalised.

Some images are grayscale or RGBA; torchvision's loader converts every image to 3-channel RGB.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Two views of the same training folder: one with augmentation, one without (for validation)
full_train_aug  = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
full_train_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_tf)
test_ds         = datasets.ImageFolder(TEST_DIR,  transform=eval_tf)
CLASSES = full_train_aug.classes
assert CLASSES == test_ds.classes
print("Classes:", CLASSES)

# Stratified train/validation split
idx = np.arange(len(full_train_aug))
train_idx, val_idx = train_test_split(idx, test_size=VAL_SPLIT, stratify=full_train_aug.targets, random_state=SEED)
train_ds = Subset(full_train_aug, train_idx)
val_ds   = Subset(full_train_eval, val_idx)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 3. Build the model (transfer learning)
- Load ResNet-50 with ImageNet weights.
- **Freeze** everything except the last residual block (`layer4`), so we keep the general image features and only adapt the high-level ones.
- Replace the final layer with `Dropout → Linear(2048 → 3)`.

In [ ]:
def build_model(num_classes, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    net = models.resnet50(weights=weights)
    for name, p in net.named_parameters():
        p.requires_grad = name.startswith("layer4")
    net.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.fc.in_features, num_classes))  # new head is trainable
    return net

model = build_model(len(CLASSES), PRETRAINED).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

## 4. Train
- **Loss:** cross-entropy with **class weights**, because Covid has more images than the other two classes.
- **Optimiser:** AdamW with a cosine learning-rate schedule.
- The weights with the best validation accuracy are kept.

In [ ]:
counts = Counter(np.array(full_train_aug.targets)[train_idx])
class_weights = torch.tensor([len(train_idx) / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                             dtype=torch.float32, device=DEVICE)
print("Class weights:", dict(zip(CLASSES, class_weights.cpu().numpy().round(3))))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc, best_state = 0.0, None
start = time.time()

for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_dl, train=True)
    vl, va = run_epoch(val_dl, train=False)
    scheduler.step()
    for k, v in zip(history, [tl, vl, ta, va]):
        history[k].append(v)
    if va >= best_acc:
        best_acc, best_state = va, copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")

print(f"\nTraining time: {(time.time()-start)/60:.1f} min | best val acc: {best_acc:.3f}")
model.load_state_dict(best_state)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, len(history["train_loss"]) + 1)
ax[0].plot(ep, history["train_loss"], label="train"); ax[0].plot(ep, history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, history["train_acc"], label="train"); ax[1].plot(ep, history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=120); plt.show()

## 5. Evaluate on the unseen test set

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_dl:
        y_pred += model(x.to(DEVICE)).argmax(1).cpu().tolist()
        y_true += y.tolist()

test_acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {test_acc:.3f}\n")
report = classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0)
print(report)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Test accuracy: {test_acc:.4f}\n\n{report}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix (test acc {test_acc:.2%})")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=120); plt.show()

In [ ]:
# Predict a single image (the same logic runs later inside the endpoint)
def predict(path):
    img = Image.open(path).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(model(eval_tf(img).unsqueeze(0).to(DEVICE)), 1)[0].cpu()
    return CLASSES[int(probs.argmax())], {c: round(float(p), 3) for c, p in zip(CLASSES, probs)}

sample = os.path.join(TEST_DIR, CLASSES[0], sorted(os.listdir(os.path.join(TEST_DIR, CLASSES[0])))[0])
print(sample, "->", predict(sample))

## 6. Save the model and register it in Azure ML
The folder `outputs/model/` holds the weights (`covid_resnet50.pt`) and a `labels.json` file with the class names and preprocessing settings. `src/score.py` reads both files when it serves predictions.

In [ ]:
torch.save(model.state_dict(), os.path.join(MODEL_DIR, "covid_resnet50.pt"))
with open(os.path.join(MODEL_DIR, "labels.json"), "w") as f:
    json.dump({"classes": CLASSES, "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
               "arch": "resnet50", "test_accuracy": round(test_acc, 4)}, f, indent=2)
print(os.listdir(MODEL_DIR))

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

try:
    ml_client = MLClient.from_config(credential=DefaultAzureCredential())
except Exception:
    # Fill these in if from_config() cannot find your workspace (Azure portal -> your ML workspace -> Overview)
    ml_client = MLClient(DefaultAzureCredential(),
                         subscription_id="<SUBSCRIPTION_ID>",
                         resource_group_name="<RESOURCE_GROUP>",
                         workspace_name="<WORKSPACE_NAME>")
print("Workspace:", ml_client.workspace_name)

In [ ]:
registered = ml_client.models.create_or_update(Model(
    path=MODEL_DIR,
    name="covid-xray-resnet50",
    type=AssetTypes.CUSTOM_MODEL,
    description="ResNet-50 transfer learning: Covid / Normal / Viral Pneumonia chest X-rays",
    tags={"framework": "pytorch", "test_accuracy": f"{test_acc:.4f}"},
))
print(f"Registered model: {registered.name} (version {registered.version})")

Training is complete and the model is registered in the Azure ML model registry.

Next, open **`02_deploy_endpoint.ipynb`** to deploy it as a REST endpoint, test it, and launch the web app.